# Очистка датасета `flats_dataset`

Таблицу `flats_dataset` собрал DAG `prepare_flats_dataset`, она лежит в личной базе.
Данные в неё попали как есть, без обработки, поэтому здесь я:

1. смотрю, что с ними не так: дубликаты, пропуски, выбросы;
2. пишу три функции, которые это исправляют;
3. применяю функции и проверяю, что после них данные в порядке.

Эти же функции лежат в модуле `plugins/steps/clean_flats.py`, его использует DAG `clean_flats_dataset`
(файл `dags/clean_flats_dataset.py`). В конце ноутбука я сверяю результат модуля с результатом здешних функций,
чтобы они точно не разошлись.

In [ ]:
import os
import sys
from urllib.parse import quote_plus

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv
from sqlalchemy import create_engine

%matplotlib inline

## Читаем данные из личной базы

Тут уже нужна личная база (`DB_DESTINATION_*`): именно в неё первый DAG записал таблицу `flats_dataset`.

In [ ]:
load_dotenv()

host = os.environ['DB_DESTINATION_HOST']
port = os.environ['DB_DESTINATION_PORT']
user = os.environ['DB_DESTINATION_USER']
password = os.environ['DB_DESTINATION_PASSWORD']
db_name = os.environ['DB_DESTINATION_NAME']

engine = create_engine(f'postgresql://{user}:{quote_plus(password)}@{host}:{port}/{db_name}')

In [ ]:
data = pd.read_sql('select * from flats_dataset', engine)
data.head()

In [ ]:
print('Размер датасета:', data.shape)
data.dtypes

In [ ]:
data.describe()

## 1. Дубликаты

`id` и `flat_id` уникальны по построению, поэтому искать дубликаты по ним бесполезно: строки не совпадут никогда.
Сравниваю строки по всем остальным колонкам. Если две квартиры совпали по всем признакам - это одно и то же
объявление, загруженное дважды.

In [ ]:
feature_cols = [col for col in data.columns if col not in ['id', 'flat_id']]

is_duplicated = data.duplicated(subset=feature_cols, keep='first')
print('Дубликатов (не считая первое вхождение):', int(is_duplicated.sum()))
print('Это {:.2%} строк'.format(is_duplicated.mean()))

In [ ]:
# keep=False показывает все вхождения, а не только повторы: так видно, что строки правда одинаковые
data[data.duplicated(subset=feature_cols, keep=False)].sort_values(feature_cols).head(10)

## 2. Пропуски

Смотрю, в каких колонках пропуски и сколько их.

In [ ]:
nulls = pd.DataFrame({'пропусков': data.isnull().sum(), 'доля': data.isnull().mean().round(4)})
nulls[nulls['пропусков'] > 0].sort_values('пропусков', ascending=False)

In [ ]:
print('Всего пропусков:', int(data.isnull().sum().sum()))
print('Строк хотя бы с одним пропуском:', int(data.isnull().any(axis=1).sum()))

## 3. Выбросы

Выбросы ищу только по непрерывным признакам: площади, высота потолков, цена, год постройки.
Идентификаторы, координаты и категориальные признаки (`building_type_int`, булевы флаги) так проверять нельзя:
у них большое значение не означает "странное".

In [ ]:
continuous_cols = ['total_area', 'living_area', 'kitchen_area', 'ceiling_height', 'price', 'build_year']
data[continuous_cols].describe().T

In [ ]:
# на boxplot выбросы видно сразу: точки далеко за "усами"
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flatten(), continuous_cols):
    sns.boxplot(x=data[col], ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

### Границы по методу IQR

IQR - это расстояние между первым и третьим квартилями: `IQR = Q3 - Q1`. Нормальными считаю значения
в интервале `[Q1 - 1.5 * IQR, Q3 + 1.5 * IQR]`, остальное - выбросы. Метод простой и не зависит от того,
нормально ли распределён признак.

In [ ]:
threshold = 1.5

bounds = []
for col in continuous_cols:
    q1 = data[col].quantile(0.25)
    q3 = data[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - threshold * iqr
    upper = q3 + threshold * iqr
    outliers = int((~data[col].between(lower, upper) & data[col].notnull()).sum())
    bounds.append({'колонка': col, 'Q1': q1, 'Q3': q3, 'нижняя': lower, 'верхняя': upper, 'выбросов': outliers})

pd.DataFrame(bounds).set_index('колонка')

### Невозможные значения

Кроме статистических выбросов бывают значения, которые невозможны по смыслу. Их стоит проверить отдельно,
потому что IQR ловит не всё: например, нулевая цена по статистике может и не быть выбросом.

In [ ]:
checks = {
    'цена меньше либо равна 0': data['price'] <= 0,
    'общая площадь меньше либо равна 0': data['total_area'] <= 0,
    'жилая площадь больше общей': data['living_area'] > data['total_area'],
    'этаж выше этажности дома': data['floor'] > data['floors_total'],
}

pd.Series({name: int(mask.sum()) for name, mask in checks.items()}, name='строк')

## Функции очистки

Пишу три функции, каждая делает одно дело:

* `fill_missing_values` - заполняет пропуски: числовые колонки медианой (она устойчива к выбросам,
  в отличие от среднего), булевы и остальные - самым частым значением. `id` и `flat_id` не трогает;
* `remove_duplicates` - удаляет строки, одинаковые по всем колонкам кроме `id` и `flat_id`, оставляя первую;
* `remove_outliers` - сначала выкидывает невозможные значения (цена и площадь должны быть больше нуля),
  потом по каждому непрерывному признаку считает границы IQR и оставляет только то, что в них попало.

Порядок применения: сначала `fill_missing_values`, потом `remove_duplicates`, потом `remove_outliers`.
Пропуски заполняю первыми не просто так: строки, которые отличались только пропущенным значением,
после заполнения становятся полностью одинаковыми, и их поймает шаг с дубликатами. Если поменять эти шаги местами,
часть дублей останется. Выбросы убираю последними: `between` считает пропуск выбросом, так что к этому моменту
пропусков быть уже не должно, а удаление строк новых дублей не создаёт.

Точно такой же код лежит в `plugins/steps/clean_flats.py`.

In [ ]:
def fill_missing_values(data):
    """Заполняет пропуски: числовые колонки медианой, булевы и остальные модой."""
    data = data.copy()

    for col in data.columns:
        # id и flat_id - это идентификаторы, их заполнять нельзя
        if col in ['id', 'flat_id'] or not data[col].isnull().any():
            continue

        if pd.api.types.is_numeric_dtype(data[col]) and not pd.api.types.is_bool_dtype(data[col]):
            data[col] = data[col].fillna(data[col].median())
        else:
            # для булевых и текстовых колонок медианы нет, берём самое частое значение
            data[col] = data[col].fillna(data[col].mode()[0])

    return data


def remove_duplicates(data):
    """Удаляет строки, которые совпадают по всем признакам (id и flat_id не считаем)."""
    # одна и та же квартира может быть выложена дважды с разными id, поэтому сравниваем только признаки
    feature_cols = [col for col in data.columns if col not in ['id', 'flat_id']]
    is_duplicated = data.duplicated(subset=feature_cols, keep='first')
    return data[~is_duplicated].reset_index(drop=True)


def remove_outliers(data, threshold=1.5):
    """Убирает строки с нулевой ценой или площадью и выбросы по методу межквартильного размаха."""
    # цена и общая площадь меньше или равные нулю - это ошибка в данных, а не выброс
    data = data[(data['price'] > 0) & (data['total_area'] > 0)]

    # выброс - значение дальше, чем на threshold межквартильных размахов от границ ящика
    for col in ['total_area', 'living_area', 'kitchen_area', 'ceiling_height', 'price', 'build_year']:
        q1 = data[col].quantile(0.25)
        q3 = data[col].quantile(0.75)
        iqr = q3 - q1
        data = data[data[col].between(q1 - threshold * iqr, q3 + threshold * iqr)]

    return data.reset_index(drop=True)

## Применяем функции

Применяю по очереди и смотрю, сколько строк уходит на каждом шаге.

In [ ]:
print('Было строк:', len(data))

cleaned = fill_missing_values(data)
print('После заполнения пропусков:', len(cleaned), '| пропусков осталось:', int(cleaned.isnull().sum().sum()))

cleaned = remove_duplicates(cleaned)
print('После удаления дубликатов:', len(cleaned))

cleaned = remove_outliers(cleaned)
print('После удаления выбросов:', len(cleaned))

print('Всего удалено: {} строк ({:.2%})'.format(len(data) - len(cleaned), 1 - len(cleaned) / len(data)))

In [ ]:
# проверяю, что после очистки проблем не осталось
assert cleaned.isnull().sum().sum() == 0, 'остались пропуски'
assert cleaned.duplicated(subset=feature_cols).sum() == 0, 'остались дубликаты'
assert (cleaned['price'] > 0).all(), 'осталась цена меньше либо равная нулю'
assert (cleaned['total_area'] > 0).all(), 'осталась площадь меньше либо равная нулю'
print('Проверки прошли, данные чистые')

In [ ]:
# минимумы и максимумы стали правдоподобными, сравните с той же таблицей до очистки
cleaned[continuous_cols].describe().T

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flatten(), continuous_cols):
    sns.boxplot(x=cleaned[col], ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

## Сверяем с модулем `plugins/steps/clean_flats.py`

DAG берёт функции не из ноутбука, а из модуля в папке `plugins` (Airflow сам добавляет её в `sys.path`).
Чтобы не получилось так, что в ноутбуке один код, а в DAG другой, прогоняю данные через модуль
и сравниваю результат с тем, что получилось выше.

In [ ]:
sys.path.append('../plugins')

from steps import clean_flats

by_module = clean_flats.fill_missing_values(data)
by_module = clean_flats.remove_duplicates(by_module)
by_module = clean_flats.remove_outliers(by_module)

assert by_module.shape == cleaned.shape, 'у модуля другой размер результата'
assert by_module.equals(cleaned), 'у модуля другие данные'
print('Модуль даёт тот же результат:', by_module.shape)

## Выводы

* Дубликаты ищем по всем колонкам кроме `id` и `flat_id`, оставляем первое вхождение.
* Пропуски в числовых колонках заполняем медианой, в булевых и категориальных - модой.
* Выбросы убираем в два приёма: сначала невозможные значения (`price > 0`, `total_area > 0`),
  потом метод IQR с порогом 1.5 по площадям, высоте потолков, цене и году постройки.
* Порядок важен: заполнение пропусков -> дубликаты -> выбросы.
* После очистки пропусков и дубликатов не осталось, разброс признаков стал разумным.

Эти три функции лежат в `plugins/steps/clean_flats.py` и вызываются в шаге `transform` DAG `clean_flats_dataset`,
который читает `flats_dataset` и сохраняет результат в таблицу `clean_flats_dataset`.

In [ ]:
engine.dispose()